In [ ]:
# Environment Detection & Setup
# V4 OPTIMIZED - 10-50x FASTER TRAINING
# Key changes:
# 1. Preprocess & cache all images ONCE (not every epoch)
# 2. Use tf.data with prefetching for GPU efficiency
# 3. GPU-accelerated augmentation
# 4. Optional mixed precision training
import sys
import os

# Detect if running in Colab or local
try:
 import google.colab
 IN_COLAB = True
 print(" Running in Google Colab")
except:
 IN_COLAB = False
 print(" Running in Local Environment")

# Mount Google Drive if in Colab
if IN_COLAB:
 from google.colab import drive
 drive.mount('/content/drive')

 # Verify GPU
 import tensorflow as tf
 gpus = tf.config.list_physical_devices('GPU')
 if gpus:
 print(f" GPU Detected: {gpus[0].name}")
 else:
 print(" No GPU detected - using CPU")
else:
 print(" Local environment - configure paths accordingly")

In [ ]:
# Essential Imports
import os
import numpy as np
import pandas as pd!pip install pydicom tqdm -q
import pydicom
from pathlib import Path
from PIL import Image
import warnings
import cv2
from tqdm import tqdm
warnings.filterwarnings('ignore')

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC

# Metrics & visualization
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# V4 OPTIMIZED CONFIGURATION
from pathlib import Path

class Config:
 """Configuration for V4 OPTIMIZED mammography classification
 
 LITERATURE REFERENCES:
 - McKinney et al., Nature 2020 (Google Health)
 - Swapna Rani et al., JATIT 2025 (ILB-BCD, 99.16% accuracy)
 - Wu et al., IEEE TMI 2019 (AUC 0.895)
 
 PERFORMANCE OPTIMIZATIONS:
 1. PREPROCESS & CACHE all images once (not every epoch!)
 2. tf.data pipeline with prefetching
 3. GPU-accelerated augmentation
 4. Optional mixed precision (FP16)
 
 EXPECTED SPEEDUP: 10-50x (days → hours)
 """

 def __init__(self):
 # Detect environment
 try:
 import google.colab
 self.IN_COLAB = True
 except:
 self.IN_COLAB = False

 # Set paths based on environment
 if self.IN_COLAB:
 self.BASE_PATH = Path('/content/drive/MyDrive/CBIS-DDSM-data')
 self.DICOM_PATH = self.BASE_PATH / 'CBIS-DDSM'
 self.CSV_PATH = self.BASE_PATH / 'csv'
 self.EDA_OUTPUT_PATH = self.BASE_PATH / 'eda_complete'
 self.OUTPUT_PATH = self.BASE_PATH / 'model_outputs_v4_optimized'
 self.CHECKPOINT_PATH = self.BASE_PATH / 'checkpoints_v4_optimized'
 self.RESULTS_PATH = self.BASE_PATH / 'results_v4_optimized'
 self.CACHE_PATH = self.BASE_PATH / 'preprocessed_cache' # NEW!
 else:
 self.BASE_PATH = Path('/home/tars/Desktop/final_project/CBIS-DDSM model training')
 self.DICOM_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/CBIS-DDSM').expanduser()
 self.CSV_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/csv').expanduser()
 self.EDA_OUTPUT_PATH = Path('~/GoogleDrive/CBIS-DDSM-data/eda_complete').expanduser()
 self.OUTPUT_PATH = self.BASE_PATH / 'local_outputs_v4_optimized'
 self.CHECKPOINT_PATH = self.BASE_PATH / 'local_checkpoints_v4_optimized'
 self.RESULTS_PATH = self.BASE_PATH / 'local_results_v4_optimized'
 self.CACHE_PATH = self.BASE_PATH / 'preprocessed_cache' # NEW!

 # Data files
 self.CALC_TRAIN_CSV = self.CSV_PATH / 'calc_case_description_train_set.csv'
 self.CALC_TEST_CSV = self.CSV_PATH / 'calc_case_description_test_set.csv'
 self.MASS_TRAIN_CSV = self.CSV_PATH / 'mass_case_description_train_set.csv'
 self.MASS_TEST_CSV = self.CSV_PATH / 'mass_case_description_test_set.csv'
 self.METADATA_CSV = self.CSV_PATH / 'metadata.csv'

 # Image parameters
 self.IMG_SIZE = (224, 224)
 self.IMG_CHANNELS = 3

 # Training parameters (unchanged from V4)
 self.BATCH_SIZE = 32
 self.STAGE1_EPOCHS = 30
 self.STAGE2A_EPOCHS = 30
 self.STAGE2B_EPOCHS = 50
 
 # Learning rates
 self.STAGE1_LR = 1e-3
 self.STAGE2A_LR = 1e-4
 self.STAGE2B_LR = 1e-6
 
 # Data split
 self.TRAIN_SPLIT = 0.70
 self.VAL_SPLIT = 0.15
 self.TEST_SPLIT = 0.15

 # Regularization
 self.L1_REGULARIZATION = 0.01
 self.L2_REGULARIZATION = 0.01
 self.DROPOUT_RATE = 0.4
 self.GRADIENT_CLIPNORM = 1.0
 self.LABEL_SMOOTHING = 0.1

 # Augmentation
 self.ROTATION_RANGE = 15
 self.ZOOM_RANGE = [0.9, 1.1]
 self.BRIGHTNESS_RANGE = [0.9, 1.1]
 self.HORIZONTAL_FLIP = True

 # Class weighting
 self.USE_CLASS_WEIGHT_IN_FIT = False
 self.MALIGNANT_WEIGHT_MULTIPLIER = 3.0
 self.FOCAL_LOSS_GAMMA = 2.0
 self.FOCAL_LOSS_ALPHA = 0.75

 # Unfreezing strategy
 self.FREEZE_FIRST_N_LAYERS = 200

 # Callbacks
 self.EARLY_STOP_PATIENCE = 20
 self.LR_REDUCE_PATIENCE = 7
 self.LR_REDUCE_FACTOR = 0.5
 self.MIN_EPOCHS_STAGE1 = 20
 self.MIN_EPOCHS_STAGE2A = 20
 self.MIN_EPOCHS_STAGE2B = 30

 # NEW: PERFORMANCE OPTIMIZATION PARAMETERS
 self.PREFETCH_BUFFER = tf.data.AUTOTUNE
 self.SHUFFLE_BUFFER = 1000
 self.NUM_PARALLEL_CALLS = tf.data.AUTOTUNE
 self.USE_MIXED_PRECISION = False # Set True for modern GPUs (RTX 20/30/40, V100, A100)
 self.FORCE_CACHE_REBUILD = False # Set True to rebuild cache

 # Create directories
 self._create_dirs()

 def _create_dirs(self):
 for path in [self.OUTPUT_PATH, self.CHECKPOINT_PATH, self.RESULTS_PATH, self.CACHE_PATH]:
 path.mkdir(parents=True, exist_ok=True)

 def print_config(self):
 print("\n" + "="*70)
 print(" V4 OPTIMIZED MAMMOGRAPHY CLASSIFICATION")
 print(" 10-50x FASTER TRAINING WITH CACHING & tf.data")
 print("="*70)
 print(f"\n Paths:")
 print(f" Environment: {'Google Colab' if self.IN_COLAB else 'Local'}")
 print(f" Cache Path: {self.CACHE_PATH}")
 print(f" Checkpoint: {self.CHECKPOINT_PATH}")
 print(f"\n Training Strategy:")
 print(f" Stage 1: {self.STAGE1_EPOCHS} epochs, LR={self.STAGE1_LR}")
 print(f" Stage 2a: {self.STAGE2A_EPOCHS} epochs, LR={self.STAGE2A_LR}")
 print(f" Stage 2b: {self.STAGE2B_EPOCHS} epochs, LR={self.STAGE2B_LR}")
 print(f"\n PERFORMANCE OPTIMIZATIONS:")
 print(f" Image caching: Preprocess ONCE, reuse forever")
 print(f" tf.data pipeline with prefetching")
 print(f" GPU-accelerated augmentation")
 print(f" {'' if self.USE_MIXED_PRECISION else ''} Mixed precision (FP16)")
 print(f"\n V4 Key Fixes:")
 print(f" class_weight in fit(): DISABLED")
 print(f" Stage 2b LR: {self.STAGE2B_LR}")
 print(f" Freeze first {self.FREEZE_FIRST_N_LAYERS} layers")
 print(f" L1/L2: {self.L1_REGULARIZATION}")
 print("="*70 + "\n")

# Initialize configuration
config = Config()
config.print_config()

# Set GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
 for gpu in gpus:
 tf.config.experimental.set_memory_growth(gpu, True)
 print(f" GPU configured with memory growth")

# Enable mixed precision if configured
if config.USE_MIXED_PRECISION:
 try:
 policy = tf.keras.mixed_precision.Policy('mixed_float16')
 tf.keras.mixed_precision.set_global_policy(policy)
 print(" Mixed precision (FP16) enabled - expect 1.5-2x speedup")
 except Exception as e:
 print(f" Mixed precision not available: {e}")

## Data Loading & Preprocessing Functions

In [ ]:
# DICOM Loading Utilities

def find_dicom_file(case_dir_name, base_path):
 """Find ROI DICOM file in case directory."""
 case_path = Path(base_path) / case_dir_name
 
 if not case_path.exists():
 return None
 
 try:
 dcm_files = list(case_path.rglob("*.dcm"))
 if not dcm_files:
 return None
 
 dcm_files = sorted(dcm_files, key=lambda x: x.name)
 # Return last file (ROI crop) if multiple exist
 return dcm_files[-1] if len(dcm_files) >= 2 else dcm_files[0]
 except:
 return None


def load_and_preprocess_dicom(dcm_path, target_size=(224, 224)):
 """
 Load DICOM and preprocess to final tensor format.
 
 This function is called ONCE per image during caching,
 NOT during training (which was the original bottleneck).
 """
 try:
 dcm = pydicom.dcmread(str(dcm_path))
 img = dcm.pixel_array.astype(np.float32)
 
 if img.size == 0:
 return None
 
 # Apply Rescale Slope and Intercept
 rescale_slope = float(getattr(dcm, 'RescaleSlope', 1.0))
 rescale_intercept = float(getattr(dcm, 'RescaleIntercept', 0.0))
 if rescale_slope!= 1.0 or rescale_intercept!= 0.0:
 img = img * rescale_slope + rescale_intercept
 
 # Apply VOI LUT or Window/Level
 voi_applied = False
 if hasattr(dcm, 'VOILUTSequence') and dcm.VOILUTSequence:
 try:
 from pydicom.pixel_data_handlers.util import apply_voi_lut
 img = apply_voi_lut(img.astype(np.float64), dcm, index=0).astype(np.float32)
 voi_applied = True
 except:
 pass
 
 if not voi_applied:
 window_center = getattr(dcm, 'WindowCenter', None)
 window_width = getattr(dcm, 'WindowWidth', None)
 
 if window_center is not None and window_width is not None:
 if hasattr(window_center, '__iter__'):
 window_center = float(window_center[0])
 else:
 window_center = float(window_center)
 if hasattr(window_width, '__iter__'):
 window_width = float(window_width[0])
 else:
 window_width = float(window_width)
 
 if window_width > 0:
 img_min = window_center - window_width / 2
 img_max = window_center + window_width / 2
 img = np.clip(img, img_min, img_max)
 img = (img - img_min) / (img_max - img_min)
 voi_applied = True
 
 # Handle Photometric Interpretation
 photometric = getattr(dcm, 'PhotometricInterpretation', 'MONOCHROME2')
 if 'MONOCHROME1' in str(photometric):
 if voi_applied:
 img = 1.0 - img
 else:
 img = np.max(img) - img
 
 # Normalize if VOI wasn't applied
 if not voi_applied:
 p1, p99 = np.percentile(img, [1, 99])
 if p99 > p1:
 img = (img - p1) / (p99 - p1)
 else:
 img_min, img_max = img.min(), img.max()
 if img_max > img_min:
 img = (img - img_min) / (img_max - img_min)
 else:
 img = np.full_like(img, 0.5)
 
 img = np.clip(img, 0.0, 1.0)
 
 # Resize
 img = cv2.resize(img, target_size, interpolation=cv2.INTER_LANCZOS4)
 
 # Convert to RGB and apply DenseNet preprocessing
 img_rgb = np.stack([img, img, img], axis=-1)
 img_rgb = img_rgb * 255.0
 img_rgb = densenet_preprocess(img_rgb)
 
 return img_rgb.astype(np.float32)
 except Exception as e:
 return None

print(" DICOM loading utilities defined")

In [ ]:
# Load Dataset Metadata

def extract_case_dir_from_path(path_str):
 """Extract case directory name from CSV path."""
 if pd.isna(path_str) or path_str == '':
 return None
 parts = str(path_str).split('/')
 return parts[0] if len(parts) >= 1 else None


def load_cbis_ddsm_metadata(config):
 """Load CBIS-DDSM dataset metadata (without loading images yet)."""
 
 print("\n" + "="*70)
 print(" LOADING CBIS-DDSM METADATA")
 print("="*70)
 
 dfs = []
 csv_files = [
 (config.CALC_TRAIN_CSV, 'calc', 'train'),
 (config.CALC_TEST_CSV, 'calc', 'test'),
 (config.MASS_TRAIN_CSV, 'mass', 'train'),
 (config.MASS_TEST_CSV, 'mass', 'test'),
 ]
 
 for csv_path, lesion_type, split in csv_files:
 if csv_path.exists():
 df = pd.read_csv(csv_path)
 df['lesion_type'] = lesion_type
 df['original_split'] = split
 dfs.append(df)
 print(f" Loaded {csv_path.name}: {len(df)} records")
 else:
 print(f" Missing {csv_path.name}")
 
 if not dfs:
 raise FileNotFoundError("No CSV files found!")
 
 combined_df = pd.concat(dfs, ignore_index=True)
 print(f"\n Total records: {len(combined_df)}")
 
 # Extract pathology
 combined_df['label'] = combined_df['pathology'].apply(
 lambda x: 1 if 'MALIGNANT' in str(x).upper() else 0
 )
 
 # Find path column
 path_col = None
 for col in ['cropped image file path', 'ROI mask file path', 'image file path']:
 if col in combined_df.columns:
 path_col = col
 print(f"\n Using path column: '{col}'")
 break
 
 if path_col is None:
 raise ValueError("Could not find image path column!")
 
 # Extract case_dir
 combined_df['case_dir'] = combined_df[path_col].apply(extract_case_dir_from_path)
 combined_df = combined_df[combined_df['case_dir'].notna()].copy()
 
 # Find valid DICOM files
 print(f"\n Validating DICOM files in: {config.DICOM_PATH}")
 
 valid_samples = []
 missing = 0
 
 for _, row in tqdm(combined_df.iterrows(), total=len(combined_df), desc="Validating"):
 dcm_path = find_dicom_file(row['case_dir'], config.DICOM_PATH)
 if dcm_path:
 valid_samples.append({
 'case_dir': row['case_dir'],
 'dcm_path': str(dcm_path),
 'label': row['label'],
 'lesion_type': row['lesion_type'],
 'pathology': row['pathology']
 })
 else:
 missing += 1
 
 print(f"\n Found {len(valid_samples)} valid samples")
 print(f" Missing {missing} samples")
 
 samples_df = pd.DataFrame(valid_samples)
 
 print(f"\n Class Distribution:")
 print(f" Benign (0): {(samples_df['label'] == 0).sum()}")
 print(f" Malignant (1): {(samples_df['label'] == 1).sum()}")
 
 return samples_df

# Load metadata
samples_df = load_cbis_ddsm_metadata(config)
print("="*70)

In [ ]:
# Split Data BEFORE Caching
# Split first so can cache train/val/test separately

# Stratified split
train_df, temp_df = train_test_split(
 samples_df, 
 test_size=(config.VAL_SPLIT + config.TEST_SPLIT),
 stratify=samples_df['label'],
 random_state=42
)

val_df, test_df = train_test_split(
 temp_df,
 test_size=config.TEST_SPLIT / (config.VAL_SPLIT + config.TEST_SPLIT),
 stratify=temp_df['label'],
 random_state=42
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"\n Data Split (Stratified):")
print(f" Training: {len(train_df)} samples")
print(f" Validation: {len(val_df)} samples")
print(f" Test: {len(test_df)} samples")

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
 n_benign = (df['label'] == 0).sum()
 n_malignant = (df['label'] == 1).sum()
 print(f" {name}: Benign={n_benign}, Malignant={n_malignant}")

In [ ]:
# CRITICAL OPTIMIZATION - PREPROCESS & CACHE
# This is THE KEY FIX for slow training!
# Instead of loading DICOMs every batch, we:
# 1. Preprocess ALL images ONCE
# 2. Cache to disk as compressed.npz
# 3. Load instantly from cache in future runs

def preprocess_and_cache(df, config, split_name):
 """
 Preprocess all images and cache to disk.
 
 First run: Takes 5-15 minutes (same as before)
 Future runs: Loads in SECONDS from cache!
 
 Returns:
 images: np.array of shape (N, 224, 224, 3)
 labels: np.array of shape (N,)
 """
 cache_file = config.CACHE_PATH / f'{split_name}_cache.npz'
 
 # Check if cache exists and is valid
 if cache_file.exists() and not config.FORCE_CACHE_REBUILD:
 print(f"\n Loading {split_name} from cache: {cache_file}")
 data = np.load(cache_file)
 images = data['images']
 labels = data['labels']
 print(f" Loaded {len(images)} images instantly!")
 return images, labels
 
 # First time: preprocess all images
 print(f"\n{'='*70}")
 print(f"⏳ PREPROCESSING {split_name.upper()} SET (One-time operation)")
 print(f"{'='*70}")
 print(f" This takes 5-15 minutes but only happens ONCE.")
 print(f" Future runs will load instantly from cache.\n")
 
 images = []
 labels = []
 failed = 0
 
 for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
 img = load_and_preprocess_dicom(row['dcm_path'], config.IMG_SIZE)
 if img is not None:
 images.append(img)
 labels.append(row['label'])
 else:
 failed += 1
 
 images = np.array(images, dtype=np.float32)
 labels = np.array(labels, dtype=np.float32)
 
 print(f"\n Preprocessed {len(images)} images")
 if failed > 0:
 print(f" Failed to load {failed} images")
 
 # Save cache
 print(f" Saving cache to {cache_file}")
 np.savez_compressed(cache_file, images=images, labels=labels)
 
 cache_size_mb = cache_file.stat().st_size / (1024 * 1024)
 print(f" Cache size: {cache_size_mb:.1f} MB")
 
 return images, labels


# Preprocess and cache all splits
print("\n" + "="*70)
print(" PREPROCESSING & CACHING (The key optimization!)")
print("="*70)

train_images, train_labels = preprocess_and_cache(train_df, config, 'train')
val_images, val_labels = preprocess_and_cache(val_df, config, 'val')
test_images, test_labels = preprocess_and_cache(test_df, config, 'test')

print(f"\n All data loaded into memory:")
print(f" Train: {train_images.shape} ({train_images.nbytes / 1e9:.2f} GB)")
print(f" Val: {val_images.shape}")
print(f" Test: {test_images.shape}")

In [ ]:
# CREATE tf.data PIPELINES
# High-performance data pipeline with:
# - Prefetching (GPU never waits for CPU)
# - GPU-accelerated augmentation
# - Parallel processing

@tf.function
def augment_batch(images):
 """
 GPU-accelerated augmentation.
 Much faster than CPU-based augmentation!
 """
 # Random horizontal flip
 images = tf.image.random_flip_left_right(images)
 
 # Random brightness (±5%)
 images = tf.image.random_brightness(images, max_delta=0.05)
 
 # Random contrast (±5%)
 images = tf.image.random_contrast(images, lower=0.95, upper=1.05)
 
 return images


def create_dataset(images, labels, config, training=True):
 """
 Create optimized tf.data.Dataset.
 
 Key optimizations:
 - Data already in memory (no disk I/O during training!)
 - Prefetching overlaps CPU/GPU work
 - GPU-accelerated augmentation
 """
 dataset = tf.data.Dataset.from_tensor_slices((images, labels))
 
 if training:
 dataset = dataset.shuffle(buffer_size=config.SHUFFLE_BUFFER)
 
 dataset = dataset.batch(config.BATCH_SIZE)
 
 if training:
 dataset = dataset.map(
 lambda x, y: (augment_batch(x), y),
 num_parallel_calls=config.NUM_PARALLEL_CALLS
 )
 
 # CRITICAL: Prefetch to overlap data loading with GPU execution
 dataset = dataset.prefetch(config.PREFETCH_BUFFER)
 
 return dataset


# Create datasets
train_dataset = create_dataset(train_images, train_labels, config, training=True)
val_dataset = create_dataset(val_images, val_labels, config, training=False)
test_dataset = create_dataset(test_images, test_labels, config, training=False)

print("\n tf.data pipelines created with:")
print(f" Prefetching (AUTOTUNE)")
print(f" GPU-accelerated augmentation")
print(f" Shuffle buffer: {config.SHUFFLE_BUFFER}")
print(f" Batch size: {config.BATCH_SIZE}")

# Calculate steps
steps_per_epoch = len(train_images) // config.BATCH_SIZE
val_steps = len(val_images) // config.BATCH_SIZE
print(f"\n Steps per epoch: {steps_per_epoch} train, {val_steps} val")

In [ ]:
# Compute Class Weights (Reference Only)

balanced_weights = compute_class_weight(
 class_weight='balanced',
 classes=np.array([0, 1]),
 y=train_labels
)

config.CLASS_WEIGHTS_REFERENCE = {
 0: balanced_weights[0],
 1: balanced_weights[1] * config.MALIGNANT_WEIGHT_MULTIPLIER
}

print("\n" + "="*70)
print(" V4 CLASS WEIGHTING STRATEGY")
print("="*70)
print(f"\n Training Class Distribution:")
print(f" Benign (0): {int((train_labels == 0).sum())} ({(train_labels == 0).mean()*100:.1f}%)")
print(f" Malignant (1): {int((train_labels == 1).sum())} ({(train_labels == 1).mean()*100:.1f}%)")
print(f"\n V4 FIX: Using ONLY Focal Loss for class weighting")
print(f" Focal Loss α: {config.FOCAL_LOSS_ALPHA}")
print(f" Focal Loss γ: {config.FOCAL_LOSS_GAMMA}")
print(f" class_weight in model.fit(): DISABLED")
print("="*70)

In [ ]:
# Define Focal Loss

@tf.function
def focal_loss_fn(y_true, y_pred, gamma=2.0, alpha=0.75):
 """Optimized focal loss with @tf.function."""
 epsilon = tf.keras.backend.epsilon()
 y_pred = tf.clip_by_value(y_pred, epsilon, 1 - epsilon)
 y_true = tf.cast(y_true, tf.float32)
 
 ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
 pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
 focal_weight = tf.pow(1 - pt, gamma)
 alpha_weight = y_true * alpha + (1 - y_true) * (1 - alpha)
 
 return tf.reduce_mean(alpha_weight * focal_weight * ce)


def focal_loss(gamma=2.0, alpha=0.5):
 """Return focal loss function for model.compile()."""
 def loss_fn(y_true, y_pred):
 return focal_loss_fn(y_true, y_pred, gamma, alpha)
 return loss_fn

print(" Focal Loss defined")
print(f" α = {config.FOCAL_LOSS_ALPHA}")
print(f" γ = {config.FOCAL_LOSS_GAMMA}")

In [ ]:
# Build Model

def build_model(config):
 """Build V4 model with literature-backed architecture."""
 
 print("\n" + "="*70)
 print(" BUILDING V4 MODEL")
 print("="*70)
 
 base_model = DenseNet121(
 include_top=False,
 weights='imagenet',
 input_shape=(config.IMG_SIZE[0], config.IMG_SIZE[1], 3),
 pooling=None
 )
 
 print(f"\n DenseNet121 base loaded ({len(base_model.layers)} layers)")
 
 base_model.trainable = False
 
 x = base_model.output
 x = layers.GlobalAveragePooling2D(name='gap')(x)
 x = layers.BatchNormalization(name='bn_1')(x)
 x = layers.Dense(
 2048,
 activation='relu',
 kernel_regularizer=regularizers.l1_l2(
 l1=config.L1_REGULARIZATION,
 l2=config.L2_REGULARIZATION
 ),
 name='dense_2048'
 )(x)
 x = layers.BatchNormalization(name='bn_2')(x)
 x = layers.Dropout(config.DROPOUT_RATE, name='dropout')(x)
 
 # Output (use float32 for mixed precision compatibility)
 output = layers.Dense(1, activation='sigmoid', dtype='float32', name='output')(x)
 
 model = models.Model(inputs=base_model.input, outputs=output, name='DenseNet121_V4')
 
 trainable = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
 total = model.count_params()
 
 print(f"\n Model Parameters:")
 print(f" Total: {total:,}")
 print(f" Trainable: {trainable:,} ({trainable/total*100:.1f}%)")
 print("="*70)
 
 return model, base_model

model, base_model = build_model(config)

In [ ]:
# Define Callbacks (Optimized)

class LightweightDiagnosticCallback(keras.callbacks.Callback):
 """
 Lightweight diagnostic callback.
 
 OPTIMIZATION: Uses metrics already computed by Keras
 instead of re-running validation (which was a bottleneck!).
 """
 
 def __init__(self, check_interval=10):
 super().__init__()
 self.check_interval = check_interval
 self.history = []
 
 def on_epoch_end(self, epoch, logs=None):
 self.history.append(logs.copy() if logs else {})
 
 if (epoch + 1) % self.check_interval == 0:
 print(f"\n{'='*50}")
 print(f" Checkpoint - Epoch {epoch + 1}")
 print(f" Val AUC: {logs.get('val_auc', 0):.4f}")
 print(f" Val Loss: {logs.get('val_loss', 0):.4f}")
 print(f" Val Recall: {logs.get('val_recall', 0):.4f} (Sensitivity)")
 print(f" Val Precision: {logs.get('val_precision', 0):.4f}")
 print(f"{'='*50}\n")


def get_callbacks(stage_name, config, min_epochs=20):
 """Get optimized callbacks."""
 return [
 ModelCheckpoint(
 str(config.CHECKPOINT_PATH / f'{stage_name}_best_model.h5'),
 monitor='val_auc',
 mode='max',
 save_best_only=True,
 verbose=1
 ),
 ModelCheckpoint(
 str(config.CHECKPOINT_PATH / f'{stage_name}_best_loss.h5'),
 monitor='val_loss',
 mode='min',
 save_best_only=True,
 verbose=0
 ),
 ReduceLROnPlateau(
 monitor='val_loss',
 factor=config.LR_REDUCE_FACTOR,
 patience=config.LR_REDUCE_PATIENCE,
 min_lr=1e-8,
 verbose=1
 ),
 EarlyStopping(
 monitor='val_auc',
 mode='max',
 patience=config.EARLY_STOP_PATIENCE,
 restore_best_weights=True,
 verbose=1,
 start_from_epoch=min_epochs
 ),
 LightweightDiagnosticCallback(check_interval=10)
 ]

print(" Optimized callbacks defined")

## Stage 1: Train Classification Head Only

In [ ]:
# STAGE 1 - Train Head Only

print("\n" + "="*70)
print(" STAGE 1: TRAIN CLASSIFICATION HEAD (Frozen Base)")
print("="*70)

base_model.trainable = False

optimizer = Adam(
 learning_rate=config.STAGE1_LR,
 clipnorm=config.GRADIENT_CLIPNORM
)

model.compile(
 optimizer=optimizer,
 loss=focal_loss(gamma=config.FOCAL_LOSS_GAMMA, alpha=config.FOCAL_LOSS_ALPHA),
 metrics=[
 BinaryAccuracy(name='accuracy'),
 Precision(name='precision'),
 Recall(name='recall'),
 AUC(name='auc')
 ]
)

print(f"\n Configuration:")
print(f" Epochs: {config.STAGE1_EPOCHS}")
print(f" Learning Rate: {config.STAGE1_LR}")
print(f" Batch Size: {config.BATCH_SIZE}")
print(f" Loss: Focal Loss (α={config.FOCAL_LOSS_ALPHA}, γ={config.FOCAL_LOSS_GAMMA})")
print("="*70 + "\n")

# Train with tf.data dataset
history_stage1 = model.fit(
 train_dataset,
 validation_data=val_dataset,
 epochs=config.STAGE1_EPOCHS,
 callbacks=get_callbacks('stage1', config, min_epochs=config.MIN_EPOCHS_STAGE1),
 verbose=1
)

print("\n" + "="*70)
print(" STAGE 1 COMPLETE")
print(f" Best val_auc: {max(history_stage1.history['val_auc']):.4f}")
print("="*70)

## Stage 2a: Gradual Unfreezing (Last Dense Block)

In [ ]:
# STAGE 2a - Unfreeze Last Dense Block

print("\n" + "="*70)
print(" STAGE 2a: GRADUAL UNFREEZING (Last Dense Block)")
print("="*70)

base_model.trainable = True

for layer in base_model.layers:
 layer.trainable = False

unfreeze_from = 'conv5_block1'
found = False
unfrozen = 0

for layer in base_model.layers:
 if unfreeze_from in layer.name or found:
 layer.trainable = True
 found = True
 unfrozen += 1

print(f"\n Layer Status:")
print(f" Frozen: {len(base_model.layers) - unfrozen}")
print(f" Unfrozen: {unfrozen}")

optimizer = Adam(
 learning_rate=config.STAGE2A_LR,
 clipnorm=config.GRADIENT_CLIPNORM
)

model.compile(
 optimizer=optimizer,
 loss=focal_loss(gamma=config.FOCAL_LOSS_GAMMA, alpha=config.FOCAL_LOSS_ALPHA),
 metrics=[
 BinaryAccuracy(name='accuracy'),
 Precision(name='precision'),
 Recall(name='recall'),
 AUC(name='auc')
 ]
)

print(f"\n Configuration:")
print(f" Epochs: {config.STAGE2A_EPOCHS}")
print(f" Learning Rate: {config.STAGE2A_LR}")
print("="*70 + "\n")

history_stage2a = model.fit(
 train_dataset,
 validation_data=val_dataset,
 epochs=config.STAGE2A_EPOCHS,
 callbacks=get_callbacks('stage2a', config, min_epochs=config.MIN_EPOCHS_STAGE2A),
 verbose=1
)

print("\n" + "="*70)
print(" STAGE 2a COMPLETE")
print(f" Best val_auc: {max(history_stage2a.history['val_auc']):.4f}")
print("="*70)

## Stage 2b: Partial Fine-Tuning

In [ ]:
# STAGE 2b - PARTIAL FINE-TUNING

print("\n" + "="*70)
print(" STAGE 2b: PARTIAL FINE-TUNING")
print("="*70)

base_model.trainable = True

frozen_count = 0
for i, layer in enumerate(base_model.layers):
 if i < config.FREEZE_FIRST_N_LAYERS:
 layer.trainable = False
 frozen_count += 1
 else:
 layer.trainable = True

trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
total_params = model.count_params()

print(f"\n V4 FIX: Partial Unfreezing")
print(f" Frozen layers: {frozen_count}")
print(f" Unfrozen layers: {len(base_model.layers) - frozen_count}")
print(f" Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

optimizer = Adam(
 learning_rate=config.STAGE2B_LR,
 clipnorm=config.GRADIENT_CLIPNORM
)

model.compile(
 optimizer=optimizer,
 loss=focal_loss(gamma=config.FOCAL_LOSS_GAMMA, alpha=config.FOCAL_LOSS_ALPHA),
 metrics=[
 BinaryAccuracy(name='accuracy'),
 Precision(name='precision'),
 Recall(name='recall'),
 AUC(name='auc')
 ]
)

print(f"\n V4 Configuration:")
print(f" Epochs: {config.STAGE2B_EPOCHS}")
print(f" Learning Rate: {config.STAGE2B_LR} (10x lower than V3!)")
print(f" Frozen layers: first {config.FREEZE_FIRST_N_LAYERS}")
print("="*70 + "\n")

history_stage2b = model.fit(
 train_dataset,
 validation_data=val_dataset,
 epochs=config.STAGE2B_EPOCHS,
 callbacks=get_callbacks('stage2b', config, min_epochs=config.MIN_EPOCHS_STAGE2B),
 verbose=1
)

print("\n" + "="*70)
print(" STAGE 2b COMPLETE")
print(f" Best val_auc: {max(history_stage2b.history['val_auc']):.4f}")
print("="*70)

## Evaluation & Threshold Optimization

In [ ]:
# Final Evaluation

print("\n" + "="*70)
print(" FINAL EVALUATION")
print("="*70)

# Load best model
best_model_path = config.CHECKPOINT_PATH / 'stage2b_best_model.h5'
if best_model_path.exists():
 model.load_weights(str(best_model_path))
 print(f" Loaded best model from {best_model_path}")

# Get predictions on test set (using cached data - instant!)
print("\n Getting predictions on test set...")
y_pred = model.predict(test_images, batch_size=config.BATCH_SIZE, verbose=1)
y_true = test_labels
y_pred = y_pred.flatten()

# Calculate AUC
auc = roc_auc_score(y_true, y_pred)
print(f"\n Test Set AUC-ROC: {auc:.4f}")

# Find optimal threshold
fpr, tpr, thresholds = roc_curve(y_true, y_pred)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"\n Optimal Threshold (Youden's J): {optimal_threshold:.3f}")

# Classification at optimal threshold
y_pred_opt = (y_pred >= optimal_threshold).astype(int)

tp = ((y_pred_opt == 1) & (y_true == 1)).sum()
tn = ((y_pred_opt == 0) & (y_true == 0)).sum()
fp = ((y_pred_opt == 1) & (y_true == 0)).sum()
fn = ((y_pred_opt == 0) & (y_true == 1)).sum()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

print(f"\n Test Set Metrics @ threshold={optimal_threshold:.3f}:")
print(f" Sensitivity: {sensitivity*100:.1f}% (target: 96.2%)")
print(f" Specificity: {specificity*100:.1f}% (target: 82.5%)")
print(f" NPV: {npv*100:.1f}% (target: 99.1%)")
print(f" FNR: {fnr*100:.1f}% (target: 3.8%)")
print(f" AUC-ROC: {auc:.4f} (target: 0.93)")

print("\n" + "="*70)

In [ ]:
# Plot ROC Curve & Training History

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
ax1 = axes[0]
ax1.plot(fpr, tpr, 'b-', label=f'V4 Model (AUC = {auc:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', label='Random')
ax1.scatter([1-specificity], [sensitivity], c='red', s=100, marker='*', 
 label=f'Optimal @ {optimal_threshold:.3f}')
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('ROC Curve - V4 Optimized Model')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Training History
ax2 = axes[1]
all_val_auc = []
all_val_auc.extend(history_stage1.history['val_auc'])
all_val_auc.extend(history_stage2a.history['val_auc'])
all_val_auc.extend(history_stage2b.history['val_auc'])

epochs = range(1, len(all_val_auc) + 1)
ax2.plot(epochs, all_val_auc, 'b-', label='Val AUC')

stage1_end = len(history_stage1.history['val_auc'])
stage2a_end = stage1_end + len(history_stage2a.history['val_auc'])
ax2.axvline(x=stage1_end, color='g', linestyle='--', alpha=0.5, label='Stage 2a Start')
ax2.axvline(x=stage2a_end, color='r', linestyle='--', alpha=0.5, label='Stage 2b Start')

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation AUC')
ax2.set_title('Training History - V4 Optimized')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(config.RESULTS_PATH / 'v4_optimized_results.png'), dpi=150)
plt.show()

print(f"\n Results saved to {config.RESULTS_PATH}")

In [ ]:
# Save Final Results

import json

results = {
 'version': 'V4_OPTIMIZED',
 'test_auc': float(auc),
 'optimal_threshold': float(optimal_threshold),
 'sensitivity': float(sensitivity),
 'specificity': float(specificity),
 'npv': float(npv),
 'fnr': float(fnr),
 'configuration': {
 'stage1_lr': config.STAGE1_LR,
 'stage2a_lr': config.STAGE2A_LR,
 'stage2b_lr': config.STAGE2B_LR,
 'focal_loss_alpha': config.FOCAL_LOSS_ALPHA,
 'focal_loss_gamma': config.FOCAL_LOSS_GAMMA,
 'freeze_first_n_layers': config.FREEZE_FIRST_N_LAYERS,
 'l1_regularization': config.L1_REGULARIZATION,
 'l2_regularization': config.L2_REGULARIZATION
 },
 'optimizations': [
 'Preprocessed & cached all images (10-50x speedup)',
 'tf.data pipeline with prefetching',
 'GPU-accelerated augmentation',
 'Lightweight diagnostic callback'
 ],
 'v4_fixes': [
 'Removed class_weight from model.fit()',
 'Reduced Stage 2b LR to 1e-6',
 f'Keep first {config.FREEZE_FIRST_N_LAYERS} layers frozen',
 'L1/L2 regularization = 0.01'
 ]
}

with open(config.RESULTS_PATH / 'v4_optimized_results.json', 'w') as f:
 json.dump(results, f, indent=2)

# Save final model
model.save(str(config.OUTPUT_PATH / 'v4_optimized_final_model.h5'))

print("\n" + "="*70)
print(" V4 OPTIMIZED RESULTS SUMMARY")
print("="*70)
print(f"\n Target vs Achieved:")
print(f" Sensitivity: {sensitivity*100:.1f}% (target: 96.2%)")
print(f" Specificity: {specificity*100:.1f}% (target: 82.5%)")
print(f" NPV: {npv*100:.1f}% (target: 99.1%)")
print(f" FNR: {fnr*100:.1f}% (target: 3.8%)")
print(f" AUC-ROC: {auc:.4f} (target: 0.93)")
print(f"\n Performance Optimizations Applied:")
for opt in results['optimizations']:
 print(f" {opt}")
print(f"\n Results saved to: {config.RESULTS_PATH}")
print(f" Model saved to: {config.OUTPUT_PATH}")
print("="*70)

## V4 Optimized Summary

### Performance Optimizations

| Optimization | Before | After | Speedup |
|-------------|--------|-------|--------|
| **Image Loading** | Every batch, every epoch | Once, cached | **10-30x** |
| **Data Pipeline** | Custom generator | tf.data + prefetch | **2-3x** |
| **Augmentation** | CPU (numpy/cv2) | GPU (tf.image) | **1.5-2x** |
| **Diagnostics** | Full validation sweep | Use Keras logs | **1.2x** |

**Total Expected Speedup: 10-50x** (days → hours)

### V4 Literature-Backed Fixes

1. Removed `class_weight` from `model.fit()` (was causing double weighting)
2. Stage 2b LR: `1e-6` (10x lower than V3)
3. Keep first 200 layers frozen in Stage 2b
4. L1/L2 regularization: `0.01` (per ILB-BCD paper)
5. Gradient clipping: `clipnorm=1.0`

### Cache Information

Cached data is stored in: `{BASE_PATH}/preprocessed_cache/`
- `train_cache.npz`
- `val_cache.npz`
- `test_cache.npz`

To rebuild cache (e.g., after changing preprocessing):
```python
config.FORCE_CACHE_REBUILD = True
```